Here is how it works:
- Full Clone and extract all config files: .yml, .yaml and related .json, .sh
- Shallow Clone from all branches: afect number commits & contributors build script or config will be only from the latest snapshop

- it sorts and index the url also save the list of random sample url indeces
- after each puase or interuption the "Cloned Repo" should be emptyied
- by a new new rerun it continues the review from the last reviewed url which is log is stored in .evn by START_NUMBER
- the sample repos are stored in "Cloned_Sample"
- this will save the sample repos as well as metrics, configs, builds and test lines
- Full Clone helps to extract full contributors and commit history
- saving metadata happend immediately so it will get lost by pause/start
Extre feature in v2.0:
- it does compare the downloaded yml files with the list from the previous step


In [3]:
import pandas as pd
import os
import subprocess
import shutil
import random
from pathlib import Path
from dotenv import load_dotenv, set_key
import requests
import stat
import re

# === CONFIGURATION ===
MAX_PROJECTS = 3582
RANDOM_SEED = 42
NUM_SAMPLES_TO_KEEP = 150
ENV_FILE = 'All_tokens.env'

# === LOAD .env ===
# === LOAD .env ===
load_dotenv(ENV_FILE)

# Load all available GitHub tokens
TOKENS = [os.getenv(f'GITHUB_TOKEN_{i}') for i in range(1, 7)]
TOKENS = [t for t in TOKENS if t]

if not TOKENS:
    raise ValueError("❌ No GitHub tokens found in All_tokens.env")

token_index = 0  # For rotation

START_NUMBER = int(os.getenv("START_NUMBER"))
SAMPLE_LIST_RAW = os.getenv("SAMPLE_LIST", "").strip()
clone_errors = []
CLONE_FAILURE_COLUMNS = ["repo_index", "repo_name", "github_url", "error_message"]


# === PATHS ===
csv_path = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_2\URL_List.csv")
base_dir = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_2")
clone_dir = base_dir / "Cloned repos"
cloned_sample_dir = base_dir / "Cloned_Sample"
yml_output_dir = base_dir / "Config Files"
commits_dir = base_dir / "Commits"
build_info_dir = base_dir / "BuildInfo"
metadata_path = base_dir / "Project_Metadata.csv"
#config_location_csv = base_dir / "Config_Location.csv"
git_metadata_dir = base_dir / "Git_Metadata"

list_of_config_path = base_dir / "List_of_Config.csv"
if list_of_config_path.exists():
    config_locations_df = pd.read_csv(list_of_config_path)
else:
    config_locations_df = pd.DataFrame(columns=[
        "html_url", "repo_name", "config_file_path", "original_rel_path", "file_name", "file_type"
    ])


# === ENSURE ALL FOLDERS EXIST ===
for path in [clone_dir, yml_output_dir, commits_dir, build_info_dir, cloned_sample_dir, git_metadata_dir]:
    path.mkdir(parents=True, exist_ok=True)
# CI_Services Lock down list
ci_patterns = {
    r'\.travis\.yml$': 'Travis_CI',
    r'\.appveyor\.yml$': 'AppVeyor',
    r'appveyor\.yml$': 'AppVeyor',
    r'circle\.yml$': 'Circle_CI',
    r'\.circleci/config\.yml$': 'Circle_CI',
    r'azure-pipelines\.yml$': 'Azure_Pipelines',
    r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub_Actions',
    r'bitbucket-pipelines\.yml$': 'Bitbucket',
    r'\.gitlab-ci\.yml$': 'GitLab',
    r'Jenkinsfile\.yml$': 'Jenkins',
    r'bitrise\.yml$': 'Bitrise',
    r'bamboo\.yml$': 'Bamboo',
    r'codeship-services\.yml$': 'Codeship',
    r'\.gocd\.yaml$': 'GoCD',
    r'\.cirrus\.yml$': 'Cirrus',
    r'wercker\.yaml$': 'Wercker',
    r'semaphore\.yml$': 'Semaphore',
    r'codemagic\.yaml$': 'Nevercode',
}


# === LOAD AND CLEAN CSV ===
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")]
df[['github_url']].to_csv(base_dir / 'Sorted_URL_List.csv', index_label='Index')

# === HANDLE SAMPLE_LIST ===
if SAMPLE_LIST_RAW:
    sample_indices_to_keep = set(map(int, SAMPLE_LIST_RAW.split(',')))
    print(f"🔁 Loaded SAMPLE_LIST from .env with {len(sample_indices_to_keep)} indices.")
else:
    random.seed(RANDOM_SEED)
    sample_indices_to_keep = set(random.sample(range(len(df)), min(NUM_SAMPLES_TO_KEEP, len(df))))
    sample_string = ",".join(map(str, sorted(sample_indices_to_keep)))
    set_key(ENV_FILE, 'SAMPLE_LIST', sample_string)
    print(f"🎲 Generated and saved new SAMPLE_LIST with {len(sample_indices_to_keep)} indices.")

# === LOAD EXISTING CONFIG LOCATIONS IF RESUMING ===
# if config_location_csv.exists():
#     config_locations_df = pd.read_csv(config_location_csv)
# else:
#     config_locations_df = pd.DataFrame(columns=["repo_name", "config_file_path", "file_type"])

# === COMMIT METADATA EXTRACTION FUNCTION ===
def extract_commit_metadata(repo_path, output_folder):
    try:
        cmd_hashes = ["git", "-C", str(repo_path), "log", "--pretty=format:%H"]
        result_hashes = subprocess.run(cmd_hashes, capture_output=True, text=True, check=True)
        commit_hashes = result_hashes.stdout.strip().split("\n")

        rows = []
        for commit in commit_hashes:
            cmd_metadata = ["git", "-C", str(repo_path), "show", "--quiet",
                            f"--pretty=format:%H|%an|%ae|%ad|%s", "--date=iso", commit]
            result_metadata = subprocess.run(cmd_metadata, capture_output=True, text=True)
            if not result_metadata.stdout:
                print(f"⚠️ Skipped malformed commit in {repo_path.name} (missing metadata)")
                continue
            parts = result_metadata.stdout.strip().split("|", maxsplit=4)
            if len(parts) < 5:
                continue

            cmd_files = ["git", "-C", str(repo_path), "show", "--name-only", "--pretty=format:", commit]
            result_files = subprocess.run(cmd_files, capture_output=True, text=True, check=True)
            changed_files = [f.strip() for f in result_files.stdout.strip().split("\n") if f.strip()]

            # normalize case to be safe
            lower_changed = [c.lower() for c in changed_files]
            count_androidTest = sum("androidtest" in c for c in lower_changed)
            count_github_workflows = sum(".github/workflows" in c for c in lower_changed)
            count_gradle = sum("build.gradle" in c for c in lower_changed)

            rows.append({
                "commit_hash": parts[0],
                "author_name": parts[1],
                "author_email": parts[2],
                "commit_date": parts[3],
                "commit_message": parts[4],
                "touches_androidTest": count_androidTest > 0,
                "count_androidTest": count_androidTest,
                "touches_github_workflows": count_github_workflows > 0,
                "count_github_workflows": count_github_workflows,
                "touches_gradle": count_gradle > 0,
                "count_gradle": count_gradle
            })

        if rows:
            df = pd.DataFrame(rows)
            output_folder.mkdir(parents=True, exist_ok=True)
            flat_filename = f"{repo_path.name}__GitMetadata++contributors_commits.csv"
            df.to_csv(output_folder / flat_filename, index=False)
            print(f"✅ Saved commit metadata: {flat_filename}")
        else:
            print(f"⚠️ No commit data for {repo_path.name}")
    except subprocess.CalledProcessError as e:
        print(f"❌ Failed to extract commit data for {repo_path.name}: {e}")

# === FUNCTION TO HANDLE READ-ONLY FILES ===
def force_remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)
    func(path)

# === FUNCTION TO GET COUNT FROM GITHUB API ===
def get_count(api_url, headers):
    per_page = 100
    page = 1
    total_items = 0

    try:
        while True:
            response = requests.get(api_url, headers=headers, params={"per_page": per_page, "page": page})
            if response.status_code != 200:
                print(f"⚠️ API error on {api_url} page {page}: {response.status_code}")
                break

            items = response.json()
            if not isinstance(items, list):
                break  # Defensive check if API doesn't return a list (e.g., rate-limited or error)
            
            total_items += len(items)
            if len(items) < per_page:
                break  # No more pages
            page += 1

    except Exception as e:
        print(f"⚠️ Failed paginating {api_url}: {e}")
    
    return total_items

review_status_rows = []
# === PROCESS EACH REPO ===
for i in range(START_NUMBER - 1, len(df)):
    url = df.iloc[i]['github_url']
    parts = url.split('/')
    if len(parts) < 5:
        continue
    username, project = parts[-2], parts[-1].replace('.git', '')
    repo_index = str(i).zfill(4)
    repo_name = f"{repo_index}.{username}.{project}"
    repo_path = clone_dir / repo_name


    print(f"\n🔍 [{i+1}/{len(df)}] Processing {repo_name}...")

    try:
        result = subprocess.run(
            ['git', 'clone', '--depth', '1', '--single-branch', url, str(repo_path)],
            check=True,
            capture_output=True,
            text=True
        )
        print("✅ Clone complete")
    except subprocess.CalledProcessError as e:
        error_message = e.stderr.strip()
        print(f"❌ Clone failed for {repo_name}")
        print(f"STDERR:\n{error_message}")

        # Save review status
        review_status_rows.append({
            "html_url": url.strip(),
            "clone_status": "no",
            "yml_detected": "no"
        })
        pd.DataFrame([review_status_rows[-1]]).to_csv(
            base_dir / "Review_Status.csv", mode='a', header=not (base_dir / "Review_Status.csv").exists(), index=False
        )
        # Prepare and append the failure row in consistent order
        clone_failure_row = {
            "repo_index": repo_index,
            "repo_name": repo_name,
            "github_url": url.strip(),
            "error_message": error_message
        }

        clone_failures_path = base_dir / "Clone_Failures.csv"
        pd.DataFrame([clone_failure_row])[CLONE_FAILURE_COLUMNS].to_csv(
            clone_failures_path, mode='a', header=not clone_failures_path.exists(), index=False
        )
        continue

        # === Detect and checkout default branch from GitHub API ===
    try:
        base_api = f"https://api.github.com/repos/{username}/{project}"
        headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
        token_index += 1
        r_branch = requests.get(base_api, headers=headers, timeout=15)
        if r_branch.status_code == 200:
            default_branch = r_branch.json().get('default_branch', 'main')
            subprocess.run(["git", "-C", str(repo_path), "checkout", default_branch],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print(f"📌 Checked out default branch: {default_branch}")
        else:
            print(f"⚠️ Could not detect default branch for {repo_name}, using current HEAD")
    except Exception as e:
        print(f"⚠️ Failed to checkout default branch for {repo_name}: {e}")


    # === Check commit count ===
    try:
        result = subprocess.run(['git', '-C', str(repo_path), 'rev-list', '--count', 'HEAD'], capture_output=True, text=True, check=True)
        local_commit_count = int(result.stdout.strip())
    except subprocess.CalledProcessError:
        local_commit_count = 0
        print(f"⚠️ Could not get commit count for {repo_name}")

    if local_commit_count > 0:
        extract_commit_metadata(repo_path, git_metadata_dir)
    else:
        print(f"⚠️ No commits to extract for {repo_name}")

    # === Scan and copy config/build files ===
    ci_keywords = ['ci', 'build', 'test', 'workflow', 'pipeline', 'instrumentation']
    config_files_found = []

    for root, _, files in os.walk(repo_path):
        for file in files:
            file_lower = file.lower()
            file_path = Path(root) / file
            rel_path = str(file_path.relative_to(repo_path)).replace("\\", "/")

            try:
                should_copy = False
                file_type = file_lower.split('.')[-1]

                  # === Determine CI Platform ===
                ci_platform = "Other"
                for pattern, platform in ci_patterns.items():
                    if re.search(pattern, rel_path, re.IGNORECASE):
                        ci_platform = platform
                        break


                # === Determine if file qualifies as config ===
                # === Only keep YAML files if they match CI pattern ===
                if file_lower.endswith(('.yml', '.yaml')):
                    matched_ci_type = None
                    for pattern, platform in ci_patterns.items():
                        if re.search(pattern, rel_path, re.IGNORECASE):
                            matched_ci_type = platform
                            break
                    if matched_ci_type:
                        should_copy = True
                        ci_platform = matched_ci_type  # Override CI platform if matched
                    else:
                        should_copy = False  # Do not copy unmatched .yml/.yaml


                elif file_lower.endswith('build.gradle'):
                    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                        content = f.read().lower()
                        if any(keyword in content for keyword in ['test', 'instrumentation']):
                            should_copy = True

                elif file_lower.endswith(('.json', '.sh')):
                    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                        content = f.read().lower()
                        if any(keyword in content for keyword in ci_keywords):
                            should_copy = True

                # === If it qualifies, copy to Config Files with custom name ===
                if should_copy:
                    # Build the flat filename
                    rel_parts = rel_path.replace("/", ".").replace("\\", ".")
                    flat_filename = f"{username}.{project}__{ci_platform}++{file}"

                    # Save to build folder
                    if file_lower.endswith('build.gradle'):
                        destination_path = build_info_dir / flat_filename
                    else:
                        destination_path = yml_output_dir / flat_filename
                    shutil.copy2(file_path, destination_path)

                    # Save config metadata (same as before)
                    config_files_found.append({
                        "html_url": url.strip().rstrip('/'),
                        "repo_name": repo_name,
                        "config_file_path": flat_filename,
                        "original_rel_path": rel_path,
                        "file_name": file,
                        "file_type": file_type
                    })


                    
            except Exception as e:
                print(f"⚠️ Could not process or copy {rel_path} in {repo_name}: {e}")

    # If no config YAML files found after scanning repo
    has_yml_match = any(f["file_type"] in ("yml", "yaml") for f in config_files_found)
    review_status_rows.append({
        "html_url": url.strip(),
        "clone_status": "yes",
        "yml_detected": "yes" if has_yml_match else "no"
    })
    pd.DataFrame([review_status_rows[-1]]).to_csv(
        base_dir / "Review_Status.csv", mode='a', header=not (base_dir / "Review_Status.csv").exists(), index=False
    )

    if config_files_found:
        config_df = pd.DataFrame(config_files_found)
        list_of_config_path = base_dir / "List_of_Config.csv"
        if list_of_config_path.exists():
            config_df.to_csv(list_of_config_path, mode='a', header=False, index=False)
        else:
            config_df.to_csv(list_of_config_path, mode='w', header=True, index=False)




            # === Fetch and save metadata + contributors ===
    try:
        headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
        token_index += 1
        base_api = f"https://api.github.com/repos/{username}/{project}"
        r = requests.get(base_api, headers=headers, timeout=30)
        data = r.json()

        metadata_row = {
            "html_url": url,
            "repo_index": repo_index,
            "repo_name": repo_name,
            "id": data.get("id"),
            "name": data.get("name"),
            "full_name": data.get("full_name"),
            "owner": data.get("owner", {}).get("login"),
            "private": data.get("private"),
            "fork": data.get("fork"),
            "created_at": data.get("created_at"),
            "updated_at": data.get("updated_at"),
            "pushed_at": data.get("pushed_at"),
            "homepage": data.get("homepage"),
            "size": data.get("size"),
            "stargazers_count": data.get("stargazers_count"),
            "watchers_count": data.get("watchers_count"),
            "language": data.get("language"),
            "forks_count": data.get("forks_count"),
            "open_issues_count": data.get("open_issues_count"),
            "license": data.get("license", {}).get("name") if data.get("license") else None,
            "topics": ", ".join(data.get("topics", [])),
            "visibility": data.get("visibility"),
            "default_branch": data.get("default_branch"),
            "has_issues": data.get("has_issues"),
            "has_projects": data.get("has_projects"),
            "has_downloads": data.get("has_downloads"),
            "has_wiki": data.get("has_wiki"),
            "has_pages": data.get("has_pages"),
            "archived": data.get("archived"),
            "disabled": data.get("disabled"),
            "allow_forking": data.get("allow_forking"),
            "is_template": data.get("is_template"),
            "web_commit_signoff_required": data.get("web_commit_signoff_required"),
            "contributors": get_count(f"{base_api}/contributors", headers),
            "pull_requests": get_count(f"{base_api}/pulls?state=all", headers),
            "commits_GitAPI": get_count(f"{base_api}/commits", headers),
            "local_commit_count": local_commit_count
        }


        metadata_df = pd.DataFrame([metadata_row])
        if metadata_path.exists():
            metadata_df.to_csv(metadata_path, mode='a', header=False, index=False)
        else:
            metadata_df.to_csv(metadata_path, mode='w', header=True, index=False)
        print("📜 Metadata saved")

        # === Save contributor names ===
        contrib_url = f"{base_api}/contributors"
        headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
        token_index += 1
        r_contrib = requests.get(contrib_url, headers=headers, timeout=30)
        if r_contrib.status_code == 200:
            contributor_logins = [c['login'] for c in r_contrib.json()]
            contributors_text = "\n".join(contributor_logins)
            # === Save contributors as single file in Config Files ===
            contributors_filename = f"{username}.{project}__Contributors++list.txt"
            contributors_path = commits_dir / contributors_filename

            with open(contributors_path, "w", encoding="utf-8") as f:
                f.write(contributors_text)

            print(f"👥 Saved contributors to: {contributors_path.name}")

        else:
            print(f"⚠️ Failed to fetch contributors for {repo_name}: {r_contrib.status_code}")

    except Exception as e:
        print(f"⚠️ Metadata or contributors error for {repo_name}: {e}")

    # === Move to Cloned_Sample or delete ===
    try:
        if i in sample_indices_to_keep:
            dest_path = cloned_sample_dir / repo_path.name
            if dest_path.exists():
                shutil.rmtree(dest_path, ignore_errors=True)
            shutil.move(str(repo_path), str(dest_path))
            print(f"📆 Sample repo moved to: {dest_path}")
        else:
            shutil.rmtree(repo_path, onerror=force_remove_readonly)
            print(f"🕵️ Deleted cloned repo: {repo_name}")
            #print(f"🕵️ Single Search cloned repo: {repo_name}")
    except Exception as e:
        print(f"❌ Error handling repo folder for {repo_name}: {e}")

    set_key(ENV_FILE, 'START_NUMBER', str(i + 2))
    #config_locations_df.drop_duplicates().to_csv(config_location_csv, index=False)


    if clone_errors:
        error_df = pd.DataFrame(clone_errors)
        error_df.to_csv(base_dir / "Clone_Failures.csv", index=False)
        print(f"❗ Saved clone failure reasons → {len(error_df)} repos")


# === FINAL DEDUPLICATION OF CONFIG FILE LOG ===
# === FINAL DEDUPLICATION OF ALL LOG FILES ===

# 1. List_of_Config.csv
list_of_config_path = base_dir / "List_of_Config.csv"
if list_of_config_path.exists():
    df_config = pd.read_csv(list_of_config_path)
    df_config.drop_duplicates().to_csv(list_of_config_path, index=False)
    print(f"🧹 Deduplicated List_of_Config.csv → {len(df_config)} rows")

# 2. Clone_Failures.csv
clone_failures_path = base_dir / "Clone_Failures.csv"
if clone_failures_path.exists():
    df_failures = pd.read_csv(clone_failures_path)
    df_failures = df_failures[CLONE_FAILURE_COLUMNS]  # Reorder if needed
    df_failures.drop_duplicates().to_csv(clone_failures_path, index=False)
    print(f"🧹 Deduplicated Clone_Failures.csv → {len(df_failures)} rows")



# 3. Project_Metadata.csv
if metadata_path.exists():
    df_metadata = pd.read_csv(metadata_path)
    df_metadata.drop_duplicates().to_csv(metadata_path, index=False)
    print(f"🧹 Deduplicated Project_Metadata.csv → {len(df_metadata)} rows")

# 4. Review_Status.csv
review_status_path = base_dir / "Review_Status.csv"
if review_status_path.exists():
    df_review = pd.read_csv(review_status_path)
    df_review.drop_duplicates().to_csv(review_status_path, index=False)
    print(f"🧹 Deduplicated Review_Status.csv → {len(df_review)} rows")



print(f"\n✅ Process complete. Sampled: {len(sample_indices_to_keep)} | Total Processed: {len(df) - (START_NUMBER - 1)}")
print("\n✅ All selected repositories have been processed.")


🎲 Generated and saved new SAMPLE_LIST with 150 indices.

🔍 [1/2960] Processing 0000.wuan.bo-android...
✅ Clone complete
📌 Checked out default branch: main
✅ Saved commit metadata: 0000.wuan.bo-android__GitMetadata++contributors_commits.csv
📜 Metadata saved
👥 Saved contributors to: wuan.bo-android__Contributors++list.txt
🕵️ Deleted cloned repo: 0000.wuan.bo-android

🔍 [2/2960] Processing 0001.MikeOrtiz.TouchImageView...
✅ Clone complete
📌 Checked out default branch: master
✅ Saved commit metadata: 0001.MikeOrtiz.TouchImageView__GitMetadata++contributors_commits.csv
📜 Metadata saved
👥 Saved contributors to: MikeOrtiz.TouchImageView__Contributors++list.txt
🕵️ Deleted cloned repo: 0001.MikeOrtiz.TouchImageView

🔍 [3/2960] Processing 0002.andstatus.andstatus...
✅ Clone complete
📌 Checked out default branch: master
✅ Saved commit metadata: 0002.andstatus.andstatus__GitMetadata++contributors_commits.csv
⚠️ API error on https://api.github.com/repos/andstatus/andstatus/commits page 12: 504
📜 Me

Exception in thread Thread-653 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1009, in _bootstrap_inner
    self.run()
  File "c:\GitHub\.PCvenv\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 946, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1494, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 92: character maps to <undefined>


📌 Checked out default branch: master
⚠️ Skipped malformed commit in 0066.fan123199.v2ex-simple (missing metadata)
⚠️ No commit data for 0066.fan123199.v2ex-simple
📜 Metadata saved
👥 Saved contributors to: fan123199.v2ex-simple__Contributors++list.txt
🕵️ Deleted cloned repo: 0066.fan123199.v2ex-simple

🔍 [68/2960] Processing 0067.KDE.kdeconnect-android...
✅ Clone complete
📌 Checked out default branch: master
✅ Saved commit metadata: 0067.KDE.kdeconnect-android__GitMetadata++contributors_commits.csv
📜 Metadata saved
👥 Saved contributors to: KDE.kdeconnect-android__Contributors++list.txt
🕵️ Deleted cloned repo: 0067.KDE.kdeconnect-android

🔍 [69/2960] Processing 0068.tasomaniac.OpenLinkWith...
✅ Clone complete
📌 Checked out default branch: master
✅ Saved commit metadata: 0068.tasomaniac.OpenLinkWith__GitMetadata++contributors_commits.csv
📜 Metadata saved
👥 Saved contributors to: tasomaniac.OpenLinkWith__Contributors++list.txt
🕵️ Deleted cloned repo: 0068.tasomaniac.OpenLinkWith

🔍 [70/296

Exception in thread Thread-911 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1009, in _bootstrap_inner
    self.run()
  File "c:\GitHub\.PCvenv\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 946, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1494, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 111: character maps to <undefined>


📌 Checked out default branch: develop
⚠️ Skipped malformed commit in 0092.caiyonglong.MusicLake (missing metadata)
⚠️ No commit data for 0092.caiyonglong.MusicLake
📜 Metadata saved
👥 Saved contributors to: caiyonglong.MusicLake__Contributors++list.txt
🕵️ Deleted cloned repo: 0092.caiyonglong.MusicLake

🔍 [94/2960] Processing 0093.kittinunf.Fuse...
✅ Clone complete
📌 Checked out default branch: master
✅ Saved commit metadata: 0093.kittinunf.Fuse__GitMetadata++contributors_commits.csv
📜 Metadata saved
👥 Saved contributors to: kittinunf.Fuse__Contributors++list.txt
🕵️ Deleted cloned repo: 0093.kittinunf.Fuse

🔍 [95/2960] Processing 0094.SecUSo.privacy-friendly-notes...
✅ Clone complete
📌 Checked out default branch: master
✅ Saved commit metadata: 0094.SecUSo.privacy-friendly-notes__GitMetadata++contributors_commits.csv
📜 Metadata saved
👥 Saved contributors to: SecUSo.privacy-friendly-notes__Contributors++list.txt
🕵️ Deleted cloned repo: 0094.SecUSo.privacy-friendly-notes

🔍 [96/2960] Proc

...

📌 Checked out default branch: master
⚠️ Skipped malformed commit in 2909.1250422131.BiliVideoTunes (missing metadata)
⚠️ No commit data for 2909.1250422131.BiliVideoTunes
📜 Metadata saved
👥 Saved contributors to: 1250422131.BiliVideoTunes__Contributors++list.txt
🕵️ Deleted cloned repo: 2909.1250422131.BiliVideoTunes

🔍 [2911/2960] Processing 2910.MoeKeyDev.MoeKey...
✅ Clone complete
📌 Checked out default branch: main
✅ Saved commit metadata: 2910.MoeKeyDev.MoeKey__GitMetadata++contributors_commits.csv
📜 Metadata saved
👥 Saved contributors to: MoeKeyDev.MoeKey__Contributors++list.txt
🕵️ Deleted cloned repo: 2910.MoeKeyDev.MoeKey

🔍 [2912/2960] Processing 2911.yatendra2001.ai_buddy...
✅ Clone complete
📌 Checked out default branch: main
✅ Saved commit metadata: 2911.yatendra2001.ai_buddy__GitMetadata++contributors_commits.csv
📜 Metadata saved
👥 Saved contributors to: yatendra2001.ai_buddy__Contributors++list.txt
🕵️ Deleted cloned repo: 2911.yatendra2001.ai_buddy

🔍 [2913/2960] Processing 

Exception in thread Thread-28374 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1009, in _bootstrap_inner
    self.run()
  File "c:\GitHub\.PCvenv\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 946, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1494, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 101: character maps to <undefined>


📌 Checked out default branch: main
⚠️ Skipped malformed commit in 2926.NonebotGUI.nonebot-flutter-gui (missing metadata)
⚠️ No commit data for 2926.NonebotGUI.nonebot-flutter-gui
📜 Metadata saved
👥 Saved contributors to: NonebotGUI.nonebot-flutter-gui__Contributors++list.txt
🕵️ Deleted cloned repo: 2926.NonebotGUI.nonebot-flutter-gui

🔍 [2928/2960] Processing 2927.Predidit.Kazumi...
✅ Clone complete
📌 Checked out default branch: main
✅ Saved commit metadata: 2927.Predidit.Kazumi__GitMetadata++contributors_commits.csv
📜 Metadata saved
👥 Saved contributors to: Predidit.Kazumi__Contributors++list.txt
🕵️ Deleted cloned repo: 2927.Predidit.Kazumi

🔍 [2929/2960] Processing 2928.canxin121.app_rhyme...
✅ Clone complete
📌 Checked out default branch: main
✅ Saved commit metadata: 2928.canxin121.app_rhyme__GitMetadata++contributors_commits.csv
📜 Metadata saved
👥 Saved contributors to: canxin121.app_rhyme__Contributors++list.txt
🕵️ Deleted cloned repo: 2928.canxin121.app_rhyme

🔍 [2930/2960] Proce

Exception in thread Thread-28694 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1009, in _bootstrap_inner
    self.run()
  File "c:\GitHub\.PCvenv\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\threading.py", line 946, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1494, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\gilla\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8d in position 124: character maps to <undefined>


📌 Checked out default branch: master
⚠️ Skipped malformed commit in 2959.FoxSensei001.LoveIwara (missing metadata)
⚠️ No commit data for 2959.FoxSensei001.LoveIwara
📜 Metadata saved
👥 Saved contributors to: FoxSensei001.LoveIwara__Contributors++list.txt
🕵️ Deleted cloned repo: 2959.FoxSensei001.LoveIwara
🧹 Deduplicated List_of_Config.csv → 25922 rows
🧹 Deduplicated Clone_Failures.csv → 78 rows
🧹 Deduplicated Project_Metadata.csv → 2881 rows
🧹 Deduplicated Review_Status.csv → 2960 rows

✅ Process complete. Sampled: 150 | Total Processed: 2960

✅ All selected repositories have been processed.
